# AutoGen 程式碼執行 Agent 教程

## 📚 簡介

本教程將深入探討 AutoGen 的程式碼執行功能，這是 AutoGen 最強大的特性之一。

### 學習目標

1. ✅ 理解 CodeExecutor 的工作原理
2. ✅ 配置安全的執行環境
3. ✅ 實現自動化程式碼生成和執行
4. ✅ 處理執行錯誤和調試
5. ✅ 掌握進階執行技巧

### 什麼是程式碼執行 Agent？

程式碼執行 Agent 可以：
- 生成 Python 程式碼
- 在隔離環境中執行程式碼
- 查看執行結果
- 根據錯誤自動修正
- 迭代優化程式碼

## 1. 安裝和準備

In [ ]:
# 安裝必要的套件
!pip install pyautogen python-dotenv -q

# 如果需要使用 Docker
# !pip install docker -q

In [ ]:
import os
from dotenv import load_dotenv
import autogen

# 加載環境變數
load_dotenv()

# 配置 LLM
config_list = [
    {
        "model": "gpt-4",
        "api_key": os.getenv("OPENAI_API_KEY")
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0.7
}

print("✅ 配置完成！")

## 2. 基礎程式碼執行

### 2.1 最簡單的程式碼執行範例

In [ ]:
# 創建 Assistant Agent（負責生成程式碼）
coder = autogen.AssistantAgent(
    name="程式設計師",
    system_message="""你是一個專業的 Python 程式設計師。
    當用戶請求時，請編寫清晰、高效的 Python 程式碼。
    程式碼應該包含註釋和錯誤處理。""",
    llm_config=llm_config
)

# 創建 User Proxy Agent（負責執行程式碼）
executor = autogen.UserProxyAgent(
    name="執行者",
    human_input_mode="NEVER",  # 不需要人工輸入
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "coding_workspace",  # 工作目錄
        "use_docker": False,  # 暫不使用 Docker
    }
)

print("✅ Agents 已創建！")

In [ ]:
# 執行簡單任務
task = """請寫一個函數來計算斐波那契數列的第 n 項，
然後計算第 15 項的值並打印出來。"""

executor.initiate_chat(
    coder,
    message=task
)

### 2.2 數學問題求解

In [ ]:
# 創建數學問題求解 Agent
math_solver = autogen.AssistantAgent(
    name="數學專家",
    system_message="""你是一個數學專家，擅長使用 Python 解決數學問題。
    請提供：
    1. 問題分析
    2. 解題思路
    3. Python 程式碼
    4. 詳細的解釋""",
    llm_config=llm_config
)

# 數學問題
math_task = """請解決以下問題：
一個圓形花園的半徑是 10 米，周圍有一條寬 2 米的小路。
請計算：
1. 花園的面積
2. 小路的面積
3. 總面積（花園 + 小路）
請使用 Python 計算並用圖表展示。
"""

executor.initiate_chat(
    math_solver,
    message=math_task
)

## 3. 安全執行環境配置

### 3.1 使用 Docker 隔離（推薦）

In [ ]:
# Docker 配置示例
# 注意：需要先安裝 Docker

docker_executor = autogen.UserProxyAgent(
    name="Docker執行者",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config={
        "work_dir": "docker_workspace",
        "use_docker": True,  # 啟用 Docker
        "docker_image": "python:3.11-slim",  # 指定 Docker 鏡像
        "timeout": 60,  # 執行超時時間（秒）
        "last_n_messages": 3,  # 限制上下文消息數量
    }
)

print("✅ Docker 執行環境已配置（需要 Docker 運行）")

### 3.2 執行限制和安全設置

In [ ]:
# 安全配置示例
safe_executor = autogen.UserProxyAgent(
    name="安全執行者",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config={
        "work_dir": "safe_workspace",
        "use_docker": False,
        "timeout": 30,  # 30 秒超時
        "last_n_messages": 2,  # 限制上下文
    },
    # 只允許執行特定類型的程式碼
    function_map={}  # 可以註冊允許的函數
)

print("✅ 安全執行環境已配置")

## 4. 數據分析與可視化

### 4.1 自動數據分析

In [ ]:
# 創建數據分析 Agent
data_analyst = autogen.AssistantAgent(
    name="數據分析師",
    system_message="""你是一個專業的數據分析師，精通 Python 數據分析工具。
    使用 pandas、numpy、matplotlib 等庫進行數據分析。
    請提供：
    1. 數據探索
    2. 統計分析
    3. 可視化圖表
    4. 洞察和建議
    
    在生成圖表時，使用 plt.savefig() 保存圖片。""",
    llm_config=llm_config
)

# 數據分析任務
analysis_task = """請生成並分析一組模擬的銷售數據：
1. 創建 100 條隨機銷售記錄（包含日期、產品、銷售額）
2. 計算基本統計指標（平均值、中位數、標準差）
3. 分析銷售趨勢
4. 繪製銷售額分佈圖和時間序列圖
5. 提供數據洞察
"""

executor.initiate_chat(
    data_analyst,
    message=analysis_task
)

### 4.2 機器學習建模

In [ ]:
# 創建 ML 工程師 Agent
ml_engineer = autogen.AssistantAgent(
    name="ML工程師",
    system_message="""你是一個機器學習工程師。
    請使用 scikit-learn 構建和評估機器學習模型。
    包括：
    1. 數據準備
    2. 特徵工程
    3. 模型訓練
    4. 模型評估
    5. 結果可視化""",
    llm_config=llm_config
)

# 機器學習任務
ml_task = """請構建一個分類模型：
1. 使用 sklearn.datasets.make_classification 生成數據集
2. 分割訓練集和測試集
3. 訓練一個決策樹分類器
4. 評估模型性能（準確率、精確率、召回率）
5. 繪製混淆矩陣
"""

executor.initiate_chat(
    ml_engineer,
    message=ml_task
)

## 5. 錯誤處理和調試

### 5.1 自動錯誤修復

In [ ]:
# 創建帶調試能力的 Agent
debugger = autogen.AssistantAgent(
    name="調試專家",
    system_message="""你是一個程式調試專家。
    當程式碼出錯時：
    1. 分析錯誤信息
    2. 找出問題根源
    3. 提供修復方案
    4. 重寫正確的程式碼
    5. 解釋修改原因
    
    請確保修復後的程式碼能夠正確執行。""",
    llm_config=llm_config
)

# 故意包含錯誤的任務
debug_task = """請寫一個程式來計算列表的平均值，
並處理可能的錯誤情況（空列表、非數字元素等）。
然後測試這個函數。
"""

executor.initiate_chat(
    debugger,
    message=debug_task
)

### 5.2 查看執行歷史

In [ ]:
# 查看對話歷史
def display_conversation_history(agent, other_agent, max_messages=5):
    """顯示對話歷史"""
    print("\n=== 對話歷史 ===")
    messages = agent.chat_messages.get(other_agent, [])
    
    for i, msg in enumerate(messages[-max_messages:], 1):
        role = msg.get("role", "unknown")
        content = msg.get("content", "")
        print(f"\n[{i}] {role.upper()}:")
        print(content[:200] + "..." if len(content) > 200 else content)
        print("-" * 50)

# 顯示歷史
display_conversation_history(executor, coder)

## 6. 進階執行技巧

### 6.1 多輪迭代優化

In [ ]:
# 創建優化專家
optimizer = autogen.AssistantAgent(
    name="優化專家",
    system_message="""你是一個程式優化專家。
    你的任務是：
    1. 編寫初始版本的程式碼
    2. 分析性能
    3. 提出優化方案
    4. 實現優化
    5. 對比優化前後的性能
    
    使用 timeit 模塊測量執行時間。""",
    llm_config=llm_config
)

# 優化任務
optimization_task = """請編寫一個計算質數的函數，並優化其性能：
1. 寫一個基礎版本的質數判斷函數
2. 測試找出 1-10000 之間所有質數的執行時間
3. 優化這個函數（提示：使用埃拉托斯特尼篩法）
4. 測試優化後的執行時間
5. 對比性能提升
"""

executor.initiate_chat(
    optimizer,
    message=optimization_task
)

### 6.2 文件操作和持久化

In [ ]:
# 創建文件處理 Agent
file_handler = autogen.AssistantAgent(
    name="文件處理專家",
    system_message="""你是一個文件處理專家。
    擅長：
    1. 讀寫各種格式的文件（CSV、JSON、TXT 等）
    2. 數據轉換和處理
    3. 文件管理
    
    請確保程式碼包含適當的錯誤處理。""",
    llm_config=llm_config
)

# 文件處理任務
file_task = """請完成以下文件操作：
1. 創建一個包含學生成績的字典數據
2. 將數據保存為 CSV 文件
3. 讀取 CSV 文件
4. 計算每個學生的平均分
5. 將結果保存為 JSON 文件
6. 列出工作目錄中的所有文件
"""

executor.initiate_chat(
    file_handler,
    message=file_task
)

### 6.3 網頁爬蟲和 API 調用

In [ ]:
# 創建網絡請求 Agent
web_scraper = autogen.AssistantAgent(
    name="網絡爬蟲專家",
    system_message="""你是一個網絡爬蟲專家。
    擅長使用 requests 和 BeautifulSoup 進行網頁抓取。
    
    注意：
    1. 遵守 robots.txt
    2. 添加適當的請求間隔
    3. 處理異常情況
    4. 提取結構化數據""",
    llm_config=llm_config
)

# 網絡請求任務（使用公開 API）
web_task = """請編寫程式來獲取和處理網絡數據：
1. 使用 requests 獲取一個公開 API 的數據（例如：https://api.github.com/users/github）
2. 解析 JSON 響應
3. 提取關鍵信息
4. 格式化輸出
5. 添加錯誤處理（網絡超時、無效響應等）
"""

executor.initiate_chat(
    web_scraper,
    message=web_task
)

## 7. 實用工具函數

### 7.1 監控程式碼執行

In [ ]:
import time
from datetime import datetime

class CodeExecutionMonitor:
    """監控程式碼執行的工具類"""
    
    def __init__(self):
        self.executions = []
    
    def log_execution(self, code, result, execution_time):
        """記錄執行信息"""
        self.executions.append({
            "timestamp": datetime.now(),
            "code": code[:100] + "..." if len(code) > 100 else code,
            "result": str(result)[:100] + "..." if len(str(result)) > 100 else str(result),
            "execution_time": execution_time
        })
    
    def get_summary(self):
        """獲取執行摘要"""
        print("\n=== 執行摘要 ===")
        print(f"總執行次數: {len(self.executions)}")
        if self.executions:
            total_time = sum(e["execution_time"] for e in self.executions)
            print(f"總執行時間: {total_time:.2f} 秒")
            print(f"平均執行時間: {total_time/len(self.executions):.2f} 秒")

# 使用示例
monitor = CodeExecutionMonitor()

# 模擬記錄
monitor.log_execution("print('Hello')", "Hello", 0.001)
monitor.log_execution("sum([1,2,3])", "6", 0.002)

monitor.get_summary()

### 7.2 程式碼品質檢查

In [ ]:
# 創建程式碼審查 Agent
code_reviewer = autogen.AssistantAgent(
    name="程式碼審查員",
    system_message="""你是一個程式碼審查專家。
    審查程式碼時考慮：
    1. 可讀性和清晰度
    2. 效率和性能
    3. 錯誤處理
    4. 安全性
    5. 最佳實踐
    
    提供具體的改進建議和重構後的程式碼。""",
    llm_config=llm_config
)

# 程式碼審查任務
review_task = """請審查並改進以下程式碼：

```python
def calc(l):
    s = 0
    for i in l:
        s = s + i
    return s / len(l)
```

請提供：
1. 程式碼問題分析
2. 改進建議
3. 重構後的程式碼
4. 測試用例
"""

executor.initiate_chat(
    code_reviewer,
    message=review_task
)

## 8. 最佳實踐總結

### ✅ 推薦做法

1. **使用 Docker 隔離**
   ```python
   code_execution_config={"use_docker": True}
   ```

2. **設置執行超時**
   ```python
   code_execution_config={"timeout": 60}
   ```

3. **限制自動回復次數**
   ```python
   max_consecutive_auto_reply=10
   ```

4. **明確終止條件**
   ```python
   is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
   ```

5. **提供清晰的 system_message**
   - 明確 Agent 的角色和職責
   - 說明期望的輸出格式
   - 包含相關的注意事項

### ❌ 避免的做法

1. ❌ 在生產環境中不使用 Docker
2. ❌ 不設置超時限制
3. ❌ 允許無限制的自動回復
4. ❌ 執行不可信的程式碼
5. ❌ 忽略錯誤處理

### 🔒 安全建議

1. 始終在隔離環境中執行程式碼
2. 限制文件系統訪問權限
3. 監控資源使用（CPU、記憶體）
4. 記錄所有執行活動
5. 定期審查生成的程式碼

## 9. 練習題

### 初級練習

1. 創建一個 Agent 來生成並執行一個簡單的計算器程式
2. 編寫程式碼來讀取和分析一個 CSV 文件
3. 實現一個數字猜謎遊戲

### 中級練習

4. 構建一個數據分析流水線（數據加載 → 清洗 → 分析 → 可視化）
5. 實現一個簡單的機器學習模型訓練流程
6. 創建一個網頁內容提取工具

### 高級練習

7. 構建一個自動化測試生成系統
8. 實現一個程式碼性能分析和優化工具
9. 創建一個多步驟的數據處理工作流

## 10. 總結

本教程介紹了 AutoGen 的程式碼執行功能：

✅ **已學習的內容**：
- CodeExecutor 的基本使用
- 安全執行環境配置
- 數據分析和可視化
- 錯誤處理和調試
- 進階執行技巧
- 最佳實踐和安全建議

🎯 **下一步**：
- 學習 [ConversableAgent詳解](3.ConversableAgent詳解.md)
- 探索 [高級功能與技巧](5.高級功能與技巧.md)
- 嘗試 [實戰項目案例](7.實戰項目案例.md)

## 📚 參考資源

- [AutoGen 官方文檔 - Code Execution](https://microsoft.github.io/autogen/docs/topics/code-execution/)
- [Docker 文檔](https://docs.docker.com/)
- [Python 安全最佳實踐](https://python.readthedocs.io/en/stable/library/security_warnings.html)